# `Passage` — a passage that knows about its own tokens

The pipeline moves passages around as lists of bare strings, so every component
that wants a lemma, a case or a part of speech asks for it again. `Passage`
gathers that knowledge once, in one structure, so it can be inspected, printed,
or handed to a model as context.

> **Nothing in the pipeline uses this.** The oracle, the teacher and the
> formulations all still take token lists. `Passage.surfaces` is the drop-in for
> them if that ever changes — but putting it in the pipeline would change what
> the model sees, and that is a decision to make on evidence, not in passing.

**Annotation is staged by cost.** Construction reads nothing. The *local* layer
— lemma, morphology, names, function words, syllables — is computed on first
access and is fast. The *lexical* layer goes to Latin WordNet over the network,
so it is opt-in and never happens by accident.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, ".")

from retexo.datasets.passage import Passage, TokenInfo

passage = Passage("arma virumque cano Troiae qui primus ab oris Italiam fato profugus")
passage

Passage('arma virumque cano Troiae qui primus ab ', 11 tokens)

## It still behaves like a passage

`str()` returns the original text, `len()` the token count, and `surfaces` the
plain list of strings the rest of the codebase expects. Indexing and iteration
give you the annotated tokens instead.

In [2]:
print("str()     :", str(passage)[:50], "...")
print("len()     :", len(passage))
print("surfaces  :", passage.surfaces[:5], "...")
print()
print("passage[1]:", passage[1])

str()     : arma virumque cano Troiae qui primus ab oris Itali ...
len()     : 11
surfaces  : ['arma', 'virumque', 'cano', 'Troiae', 'qui'] ...



passage[1]: TokenInfo(index=1, surface='virumque', normalized='uirumque', lemma='uir', pos='n', morpho_tag=None, features=None, readings=None, is_name=False, is_stopword=False, enclitic=('virum', 'que'), syllables=('vi', 'rum', 'que'), synonyms=None, hypernyms=None, hyponyms=None, antonyms=None, derivatives=None, has_vector=None)


## The table view

One row per token. This is the whole local layer at a glance.

In [3]:
passage.to_frame()

,surface,lemma,pos,features,tag,readings,name,func,enclitic,syllables
i,,,,,,,,,,
0,arma,arma,n,pl.nom.,--p----n-,pl.nom. | pl.voc. | pl.acc.,,,—,ar-ma
1,virumque,uir,n,—,—,—,,,virum+que,vi-rum-que
2,cano,cano,v,1.sg.pres.ind.act.,v1spia---,1.sg.pres.ind.act.,,,—,ca-no
3,Troiae,troius,n,—,—,—,yes,,—,Tro-i-ae
4,qui,qui,n,—,—,—,,yes,—,qui
5,primus,primus,n,sg.nom.,--s---mn-,sg.nom.,,,—,pri-mus
6,ab,ab,n,—,—,—,,yes,—,ab
7,oris,ora,n,pl.dat.,--p----d-,pl.dat. | pl.abl.,,,—,o-ris
8,Italiam,italia,n,—,—,—,yes,,—,I-ta-li-am


Three things to read out of that table, because each is a real property of
Latin rather than a quirk of the code:

- **`virumque` has no features but does have an enclitic.** Collatinus cannot
  analyse it, because it is two words glued together. The enclitic column
  recovers *virum* + *que* — the split CLTK's own tokenizer promises and does
  not deliver.
- **`Troiae` lemmatizes to *troius*, an adjective.** Backoff lemmatization has
  no syntax to work with, so an ambiguous proper name resolves by frequency.
  The `name` column catches it anyway, from a different resource.
- **`qui` and `ab` are marked as function words**, which is what `ADAPT` would
  need in order to find where content substitutions concentrate.

## Ambiguity is kept, not resolved

Latin forms are richly ambiguous. `features` commits to one reading because a
detector needs a single answer; `readings` keeps them all, which is the more
honest thing to hand a model.

In [4]:
for token in passage:
    if token.readings and len(token.readings) > 1:
        print(f"{token.surface:<10} {len(token.readings)} readings: {' | '.join(token.readings)}")

arma       3 readings: pl.nom. | pl.voc. | pl.acc.
oris       2 readings: pl.dat. | pl.abl.
fato       2 readings: sg.dat. | sg.abl.


*arma* is nominative, vocative or accusative plural; *oris* is dative or
ablative plural of *ora* — and, in truth, also genitive singular of *os*, which
Collatinus does not offer here because it was asked about one lemma.

## One token in full

`TokenInfo` is a frozen dataclass. `None` means *not established* — the resource
was missing, or could not analyse this form — which is deliberately different
from an empty tuple, meaning *asked, and the answer was nothing*.

In [5]:
token = passage[0]
for field, value in vars(token).items():
    print(f"   {field:<12} {value}")

   index        0
   surface      arma
   normalized   arma
   lemma        arma
   pos          n
   morpho_tag   --p----n-
   features     pl.nom.
   readings     ('pl.nom.', 'pl.voc.', 'pl.acc.')
   is_name      False
   is_stopword  False
   enclitic     None
   syllables    ('ar', 'ma')
   synonyms     None
   hypernyms    None
   hyponyms     None
   antonyms     None
   derivatives  None
   has_vector   None


## The lexical layer, on request

`annotate()` adds what costs something. `wordnet=True` is one lookup per
distinct lemma, cached on disk between runs; `vectors=True` only records
whether a lemma has a vector at all, which is what decides whether `SYN-DIST`
could ever fire on it.

In [6]:
passage.annotate(wordnet=True, vectors=True)

for token in passage[:5]:
    print(f"{token.surface:<10} syn={len(token.synonyms or ()):<3} "
          f"hyper={len(token.hypernyms or ()):<3} "
          f"deriv={len(token.derivatives or ()):<3} vector={token.has_vector}")

arma       syn=0   hyper=14  deriv=2   vector=False
virumque   syn=0   hyper=23  deriv=14  vector=False
cano       syn=0   hyper=20  deriv=22  vector=False
Troiae     syn=0   hyper=0   deriv=0   vector=False
qui        syn=0   hyper=0   deriv=13  vector=False


In [7]:
# Derivational relations are the ones only recently made available.
for token in passage:
    if token.derivatives:
        print(f"{token.surface} ({token.lemma}) -> {', '.join(token.derivatives[:8])}")

arma (arma) -> armo, inermis
virumque (uir) -> centumuir, decemuir, duumuir, octouir, quinqueuir, semiuir, septemuir, sexuir
cano (cano) -> bucina, canor, canto, cantus, carmen, concino, incino, intercino
qui (qui) -> alioqui, aliqui, cuius, cum, quam, quicumque, quidam, quin
primus (primus) -> primarius, primo, primogenitus, princeps
fato (fatum) -> fatalis


Two cautions visible here. Many `synonyms` lists come back **empty** — not
because the word has no synonyms, but because the cached record discards any
relation list longer than 24 entries, on the grounds that a list that long is
describing a semantic field rather than a relation. And the `hypernyms` that do
survive are coarse: Latin WordNet resolves a relation to *every* lemma of the
target synset.

This is the ~30% coverage ceiling that sends most of the oracle's decisions to
insertion and deletion, seen from the other side.

## As model context

`render()` writes the passage for a model to read rather than for a terminal to
display: one token per line, fields named, and anything unestablished simply
absent rather than filled with a placeholder that would need explaining.

In [8]:
print(passage.render())

PASSAGE: arma virumque cano Troiae qui primus ab oris Italiam fato profugus
TOKENS: 11

0  arma  pos=n  morph=pl.nom. or pl.voc. or pl.acc.
1  virumque  lemma=uir  pos=n  enclitic=virum+que
2  cano  pos=v  morph=1.sg.pres.ind.act.
3  Troiae  lemma=troius  pos=n  proper-name
4  qui  pos=n  function-word
5  primus  pos=n  morph=sg.nom.
6  ab  pos=n  function-word
7  oris  lemma=ora  pos=n  morph=pl.dat. or pl.abl.
8  Italiam  lemma=italia  pos=n  proper-name
9  fato  lemma=fatum  pos=n  morph=sg.dat. or sg.abl.
10  profugus  pos=n  morph=sg.nom.


In [9]:
# With the lexical layer, trimmed so the context stays a usable size.
print(Passage("cano Troiae fato").annotate(wordnet=True).render(relations=True, max_relations=5))

PASSAGE: cano Troiae fato
TOKENS: 3

0  cano  pos=v  morph=1.sg.pres.ind.act.
      hyper: adcuro, carnificor, carnufico, desalto, executo (+15)
      deriv: bucina, canor, canto, cantus, carmen (+17)
1  Troiae  lemma=troius  pos=n  proper-name
2  fato  lemma=fatum  pos=n  morph=sg.dat. or sg.abl.
      syn: absegmen, apotelesma, apsegmen, censum, euentus (+5)
      hyper: actum, euentus, expertus, incidentia, infectus
      hypo: conexum, connexum
      deriv: fatalis


## What do we actually know?

The point of gathering this in one place is that coverage becomes measurable.
`summary()` says how much of the passage each resource could speak to — which
is the number that decides whether feeding this to a model is worth the tokens.

In [10]:
summary = passage.summary()
for key, value in summary.items():
    print(f"   {key:<18} {value}")

   tokens             11
   lemmatized         11
   with_features      6
   proper_names       2
   function_words     2
   with_enclitic      1
   feature_coverage   0.5454545454545454


In [11]:
# Coverage varies a lot by passage. Prose fares better than verse.
samples = {
    "Aeneid 1.1-3 (verse)": "arma virumque cano Troiae qui primus ab oris",
    "Caesar (prose)": "Gallia est omnis divisa in partes tres",
    "Vulgate (prose)": "in principio creavit Deus caelum et terram",
}
for label, text in samples.items():
    s = Passage(text).summary()
    print(f"{label:<24} tokens={s['tokens']:<3} "
          f"features={s['feature_coverage']:.0%}  "
          f"names={s['proper_names']}  function={s['function_words']}")

Aeneid 1.1-3 (verse)     tokens=8   features=50%  names=1  function=2


Caesar (prose)           tokens=7   features=57%  names=1  function=2


Vulgate (prose)          tokens=7   features=71%  names=1  function=2


## Limits worth knowing before this is used for anything

- **Morphological coverage is partial and uneven.** Collatinus cannot decline
  every lemma, and a form it cannot analyse simply has no features. Verse fares
  worse than prose.
- **Analysis is per token, with no syntax.** There is no dependency parse here,
  so nothing knows that an adjective agrees with its noun — which is exactly
  what would disambiguate *arma* between its three readings. That would need
  CLTK's Stanza pipeline and a model download.
- **Proper-name lemmas are unreliable** (*Troiae* → *troius*), so anything
  keyed on the lemma of a name — including vector lookups — will miss.
- **The lexical layer is a network call** the first time a lemma is seen. Fine
  for a passage, not for a corpus, unless the cache is warmed first.

## If this ever goes into the pipeline

The thing to measure first is whether any of it helps a model that already sees
the raw text. A contextual encoder has learned a good deal of Latin morphology
implicitly; handing it a lemma and a case may add nothing, or may add noise in
the places where our resources are wrong — and `Troiae → troius` shows they are
sometimes confidently wrong. The honest experiment is a token-classifier run
with and without these fields appended, scored against the same degenerate
floor as everything else.